RQ: Does learning the ordered sequence of prior agent actions provide predictive information about failure type beyond the semantics of the current message?

In [11]:
import pandas as pd
import numpy as np

train_events = pd.read_csv(
    "../data/processed/trajectory_events_train.csv"
)

test_events = pd.read_csv(
    "../data/processed/trajectory_events_test.csv"
)

targets = pd.read_csv(
    "../data/processed/trajectory_targets.csv"
)

train_targets = (
    targets[
        targets["split"] == "train"
    ]
    .reset_index(drop=True)
)

test_targets = (
    targets[
        targets["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("Train events:", train_events.shape)
print("Test events:", test_events.shape)

print("Train targets:", train_targets.shape)
print("Test targets:", test_targets.shape)

assert len(train_targets) == 1489
assert len(test_targets) == 287

Train events: (3792, 40)
Test events: (799, 40)
Train targets: (1489, 14)
Test targets: (287, 14)


In [12]:
train_groups = set(
    train_targets["canonical_group"]
)

test_groups = set(
    test_targets["canonical_group"]
)

print("Train groups:", len(train_groups))
print("Test groups:", len(test_groups))
print("Overlap:", len(train_groups & test_groups))

assert len(train_groups & test_groups) == 0

Train groups: 335
Test groups: 84
Overlap: 0


In [13]:
print(
    train_events[
        "event_role"
    ].value_counts()
)

print(
    train_events[
        "primary_tool"
    ].value_counts().head(30)
)

print(
    train_targets[
        "history_event_count"
    ].describe(
        percentiles=[
            .5,
            .9,
            .95,
            .99,
        ]
    )
)

event_role
TOOL_CALL    2256
ASSISTANT    1536
Name: count, dtype: int64
primary_tool
NO_TOOL                        1747
get_details_by_id               300
search                          107
startEngine                      72
I                                70
get_customer_by_phone            69
get_order_details                68
get_user_details                 61
get_reservation_details          55
cd                               42
get_data_usage                   42
echo                             36
ls                               34
get_flight_cost                  31
displayCarStatus                 28
fillFuelTank                     28
search_direct_flight             27
transfer_to_human_agents         26
pressBrakePedal                  25
get_nearest_airport_by_city      25
find_user_id_by_name_zip         24
book_flight                      24
book_reservation                 23
touch                            23
lockDoors                        22
retrieve_invoi

In [14]:
MAX_HISTORY = 96

In [15]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
NO_HISTORY_TOKEN = "<NO_HISTORY>"


def build_vocab(values):

    unique = sorted(
        set(
            pd.Series(values)
            .fillna("UNKNOWN")
            .astype(str)
        )
    )

    vocab = {
        PAD_TOKEN: 0,
        UNK_TOKEN: 1,
        NO_HISTORY_TOKEN: 2,
    }

    for value in unique:

        if value not in vocab:
            vocab[value] = len(vocab)

    return vocab

In [16]:
role_vocab = build_vocab(
    train_events["event_role"]
)

tool_vocab = build_vocab(
    train_events["primary_tool"]
)

print("Role vocab:", len(role_vocab))
print("Tool vocab:", len(tool_vocab))

print(role_vocab)

Role vocab: 5
Tool vocab: 149
{'<PAD>': 0, '<UNK>': 1, '<NO_HISTORY>': 2, 'ASSISTANT': 3, 'TOOL_CALL': 4}


In [17]:
def encode_category(
    value,
    vocab,
):
    value = str(value)

    return vocab.get(
        value,
        vocab[UNK_TOKEN],
    )

In [19]:
event_numeric_features = [
    "char_length",
    "word_count",
    "has_error_signal",
]

In [20]:
numeric_mean = (
    train_events[
        event_numeric_features
    ]
    .fillna(0)
    .mean()
)

numeric_std = (
    train_events[
        event_numeric_features
    ]
    .fillna(0)
    .std()
    .replace(0, 1)
)

print(numeric_mean)
print(numeric_std)

char_length         291.365506
word_count           42.074895
has_error_signal      0.060654
dtype: float64
char_length         393.255354
word_count           65.478517
has_error_signal      0.238726
dtype: float64


In [21]:
def build_event_lookup(events_df):

    lookup = {}

    grouped = events_df.groupby(
        "canonical_group",
        sort=False,
    )

    for group, df in grouped:

        lookup[group] = (
            df
            .sort_values(
                "message_index"
            )
            .reset_index(drop=True)
        )

    return lookup


train_event_lookup = build_event_lookup(
    train_events
)

test_event_lookup = build_event_lookup(
    test_events
)

print(
    len(train_event_lookup),
    len(test_event_lookup),
)

335 84


In [22]:
def build_history(
    target,
    event_lookup,
):

    group = target["canonical_group"]

    target_index = int(
        target["message_index"]
    )

    events = event_lookup[group]

    history = events[
        events["message_index"]
        < target_index
    ].copy()

    history = history.sort_values(
        "message_index"
    )

    # Keep most recent events only if necessary.
    history = history.tail(
        MAX_HISTORY
    )

    return history

In [23]:
example_target = (
    train_targets.iloc[100]
)

example_history = build_history(
    example_target,
    train_event_lookup,
)

print(
    example_target[
        [
            "canonical_group",
            "message_index",
            "failure_family",
        ]
    ]
)

display(
    example_history[
        [
            "message_index",
            "event_role",
            "primary_tool",
            "content",
        ]
    ]
)

canonical_group              a_92
message_index                   2
failure_family     tool_use_error
Name: 100, dtype: object


,message_index,event_role,primary_tool,content


In [24]:
computed_lengths = []

for _, target in (
    train_targets.iterrows()
):

    history = build_history(
        target,
        train_event_lookup,
    )

    computed_lengths.append(
        len(history)
    )

computed_lengths = np.array(
    computed_lengths
)

print(
    pd.Series(
        computed_lengths
    ).describe()
)

print(
    "Zero history:",
    (computed_lengths == 0).sum()
)

count    1489.000000
mean       15.756884
std        18.148802
min         0.000000
25%         5.000000
50%         9.000000
75%        18.000000
max        93.000000
dtype: float64
Zero history: 56


In [25]:
from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [26]:
train_current_texts = (
    train_targets["content"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_current_texts = (
    test_targets["content"]
    .fillna("")
    .astype(str)
    .tolist()
)

In [27]:
X_semantic_train = semantic_model.encode(
    train_current_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_semantic_test = semantic_model.encode(
    test_current_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(
    X_semantic_train.shape,
    X_semantic_test.shape,
)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

(1489, 384) (287, 384)


In [28]:
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
)

In [29]:
class TrajectoryDataset(Dataset):

    def __init__(
        self,
        targets_df,
        event_lookup,
        semantic_embeddings,
        role_vocab,
        tool_vocab,
        numeric_mean,
        numeric_std,
    ):

        self.targets = (
            targets_df
            .reset_index(drop=True)
        )

        self.event_lookup = (
            event_lookup
        )

        self.semantic_embeddings = (
            np.asarray(
                semantic_embeddings,
                dtype=np.float32,
            )
        )

        self.role_vocab = role_vocab
        self.tool_vocab = tool_vocab

        self.numeric_mean = (
            numeric_mean
        )

        self.numeric_std = (
            numeric_std
        )

    def __len__(self):

        return len(
            self.targets
        )

    def __getitem__(
        self,
        idx,
    ):

        target = (
            self.targets.iloc[idx]
        )

        history = build_history(
            target,
            self.event_lookup,
        )

        # ----------------------------------------------------
        # No-history special event
        # ----------------------------------------------------

        if len(history) == 0:

            role_ids = [
                self.role_vocab[
                    NO_HISTORY_TOKEN
                ]
            ]

            tool_ids = [
                self.tool_vocab[
                    NO_HISTORY_TOKEN
                ]
            ]

            numeric = np.zeros(
                (
                    1,
                    len(
                        event_numeric_features
                    ),
                ),
                dtype=np.float32,
            )

        else:

            role_ids = [
                encode_category(
                    value,
                    self.role_vocab,
                )
                for value
                in history["event_role"]
            ]

            tool_ids = [
                encode_category(
                    value,
                    self.tool_vocab,
                )
                for value
                in history["primary_tool"]
            ]

            numeric_df = (
                history[
                    event_numeric_features
                ]
                .fillna(0)
                .astype(float)
            )

            numeric_df = (
                numeric_df
                - self.numeric_mean
            ) / self.numeric_std

            numeric = (
                numeric_df
                .to_numpy(
                    dtype=np.float32
                )
            )

        return {
            "role_ids":
                torch.tensor(
                    role_ids,
                    dtype=torch.long,
                ),

            "tool_ids":
                torch.tensor(
                    tool_ids,
                    dtype=torch.long,
                ),

            "numeric":
                torch.tensor(
                    numeric,
                    dtype=torch.float32,
                ),

            "semantic":
                torch.tensor(
                    self.semantic_embeddings[
                        idx
                    ],
                    dtype=torch.float32,
                ),

            "label":
                torch.tensor(
                    int(
                        target[
                            "family_label"
                        ]
                    ),
                    dtype=torch.long,
                ),

            "group":
                target[
                    "canonical_group"
                ],
        }

In [30]:
from torch.nn.utils.rnn import (
    pad_sequence,
)

In [31]:
def trajectory_collate(
    batch,
):

    lengths = torch.tensor(
        [
            len(item["role_ids"])
            for item in batch
        ],
        dtype=torch.long,
    )

    role_ids = pad_sequence(
        [
            item["role_ids"]
            for item in batch
        ],
        batch_first=True,
        padding_value=0,
    )

    tool_ids = pad_sequence(
        [
            item["tool_ids"]
            for item in batch
        ],
        batch_first=True,
        padding_value=0,
    )

    numeric = pad_sequence(
        [
            item["numeric"]
            for item in batch
        ],
        batch_first=True,
        padding_value=0.0,
    )

    semantic = torch.stack(
        [
            item["semantic"]
            for item in batch
        ]
    )

    labels = torch.stack(
        [
            item["label"]
            for item in batch
        ]
    )

    return {
        "role_ids":
            role_ids,

        "tool_ids":
            tool_ids,

        "numeric":
            numeric,

        "lengths":
            lengths,

        "semantic":
            semantic,

        "labels":
            labels,
    }

In [32]:
from sklearn.model_selection import (
    GroupShuffleSplit,
)

inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

inner_train_idx, val_idx = next(
    inner_split.split(
        train_targets,
        train_targets[
            "family_label"
        ],
        groups=train_targets[
            "canonical_group"
        ],
    )
)

inner_train_targets = (
    train_targets
    .iloc[inner_train_idx]
    .reset_index(drop=True)
)

val_targets = (
    train_targets
    .iloc[val_idx]
    .reset_index(drop=True)
)

X_semantic_inner_train = (
    X_semantic_train[
        inner_train_idx
    ]
)

X_semantic_val = (
    X_semantic_train[
        val_idx
    ]
)

print(
    len(inner_train_targets),
    len(val_targets),
)

assert not (
    set(
        inner_train_targets[
            "canonical_group"
        ]
    )
    &
    set(
        val_targets[
            "canonical_group"
        ]
    )
)

1185 304


In [33]:
inner_train_dataset = TrajectoryDataset(
    inner_train_targets,
    train_event_lookup,
    X_semantic_inner_train,
    role_vocab,
    tool_vocab,
    numeric_mean,
    numeric_std,
)

val_dataset = TrajectoryDataset(
    val_targets,
    train_event_lookup,
    X_semantic_val,
    role_vocab,
    tool_vocab,
    numeric_mean,
    numeric_std,
)

test_dataset = TrajectoryDataset(
    test_targets,
    test_event_lookup,
    X_semantic_test,
    role_vocab,
    tool_vocab,
    numeric_mean,
    numeric_std,
)

In [34]:
BATCH_SIZE = 32

train_loader = DataLoader(
    inner_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=trajectory_collate,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=trajectory_collate,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=trajectory_collate,
)

In [35]:
import torch.nn as nn

from torch.nn.utils.rnn import (
    pack_padded_sequence,
)

In [36]:
class TrajectoryEncoder(nn.Module):

    def __init__(
        self,
        num_roles,
        num_tools,
        numeric_dim,
        role_dim=8,
        tool_dim=24,
        hidden_dim=128,
        dropout=0.20,
    ):

        super().__init__()

        self.role_embedding = (
            nn.Embedding(
                num_roles,
                role_dim,
                padding_idx=0,
            )
        )

        self.tool_embedding = (
            nn.Embedding(
                num_tools,
                tool_dim,
                padding_idx=0,
            )
        )

        input_dim = (
            role_dim
            + tool_dim
            + numeric_dim
        )

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(
        self,
        role_ids,
        tool_ids,
        numeric,
        lengths,
    ):

        role_emb = (
            self.role_embedding(
                role_ids
            )
        )

        tool_emb = (
            self.tool_embedding(
                tool_ids
            )
        )

        x = torch.cat(
            [
                role_emb,
                tool_emb,
                numeric,
            ],
            dim=-1,
        )

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        _, hidden = self.gru(
            packed
        )

        trajectory_embedding = (
            hidden[-1]
        )

        return self.dropout(
            trajectory_embedding
        )

In [37]:
NUM_CLASSES = 5
SEMANTIC_DIM = 384
TRAJECTORY_DIM = 128

In [38]:
class SemanticOnlyModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(
                SEMANTIC_DIM,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(
                128,
                NUM_CLASSES,
            ),
        )

    def forward(
        self,
        batch,
    ):

        return self.classifier(
            batch["semantic"]
        )

In [39]:
class TrajectoryOnlyModel(nn.Module):

    def __init__(
        self,
        num_roles,
        num_tools,
    ):

        super().__init__()

        self.encoder = TrajectoryEncoder(
            num_roles=num_roles,
            num_tools=num_tools,
            numeric_dim=len(
                event_numeric_features
            ),
            hidden_dim=TRAJECTORY_DIM,
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                TRAJECTORY_DIM,
                64,
            ),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(
                64,
                NUM_CLASSES,
            ),
        )

    def forward(
        self,
        batch,
    ):

        trajectory = self.encoder(
            batch["role_ids"],
            batch["tool_ids"],
            batch["numeric"],
            batch["lengths"],
        )

        return self.classifier(
            trajectory
        )

In [56]:
class SemanticTrajectoryModel(nn.Module):

    def __init__(
        self,
        num_roles,
        num_tools,
    ):

        super().__init__()

        self.encoder = TrajectoryEncoder(
            num_roles=num_roles,
            num_tools=num_tools,
            numeric_dim=len(
                event_numeric_features
            ),
            hidden_dim=TRAJECTORY_DIM,
        )

        fused_dim = (
            SEMANTIC_DIM
            + TRAJECTORY_DIM
        )

        self.classifier = nn.Sequential(
            nn.Linear(
                fused_dim,
                256,
            ),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(
                256,
                64,
            ),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(
                64,
                NUM_CLASSES,
            ),
        )

    def forward(
        self,
        batch,
    ):

        trajectory = self.encoder(
            batch["role_ids"],
            batch["tool_ids"],
            batch["numeric"],
            batch["lengths"],
        )

        fused = torch.cat(
            [
                batch["semantic"],
                trajectory,
            ],
            dim=1,
        )

        return self.classifier(
            fused
        )

In [40]:
from sklearn.utils.class_weight import (
    compute_class_weight,
)

inner_y = (
    inner_train_targets[
        "family_label"
    ]
    .to_numpy()
)

classes = np.arange(
    NUM_CLASSES
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=inner_y,
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
)

print(class_weights)

tensor([0.4523, 1.0441, 1.1791, 1.1618, 8.1724])


In [41]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

cpu


In [42]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

In [43]:
def move_batch(
    batch,
    device,
):

    return {
        key: (
            value.to(device)
            if torch.is_tensor(value)
            else value
        )
        for key, value
        in batch.items()
    }

In [44]:
@torch.no_grad()
def evaluate_model(
    model,
    loader,
):

    model.eval()

    true_labels = []
    predictions = []

    for batch in loader:

        batch = move_batch(
            batch,
            device,
        )

        logits = model(
            batch
        )

        pred = (
            logits.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )

        y = (
            batch["labels"]
            .cpu()
            .numpy()
        )

        predictions.extend(
            pred
        )

        true_labels.extend(
            y
        )

    y_true = np.asarray(
        true_labels
    )

    y_pred = np.asarray(
        predictions
    )

    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,
    }

In [48]:
import copy

In [49]:
def train_model(
    model,
    train_loader,
    val_loader,
    epochs=30,
    lr=1e-3,
    patience=5,
):

    model = model.to(
        device
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(
            device
        )
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    best_state = None
    best_macro_f1 = -np.inf

    patience_counter = 0
    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        losses = []

        for batch in train_loader:

            batch = move_batch(
                batch,
                device,
            )

            optimizer.zero_grad()

            logits = model(
                batch
            )

            loss = criterion(
                logits,
                batch["labels"],
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            losses.append(
                loss.item()
            )

        val_metrics = evaluate_model(
            model,
            val_loader,
        )

        row = {
            "epoch": epoch,
            "train_loss":
                np.mean(losses),
            "val_accuracy":
                val_metrics["accuracy"],
            "val_balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ],
            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],
            "val_weighted_f1":
                val_metrics[
                    "weighted_f1"
                ],
        }

        history.append(
            row
        )

        print(
            f"Epoch {epoch:02d} | "
            f"loss={row['train_loss']:.4f} | "
            f"val_macro_f1="
            f"{row['val_macro_f1']:.4f}"
        )

        if (
            val_metrics["macro_f1"]
            > best_macro_f1
        ):

            best_macro_f1 = (
                val_metrics[
                    "macro_f1"
                ]
            )

            best_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            patience_counter
            >= patience
        ):

            print(
                "Early stopping."
            )

            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )

In [50]:
torch.manual_seed(42)
np.random.seed(42)

semantic_only_model = (
    SemanticOnlyModel()
)

semantic_only_model, semantic_history = (
    train_model(
        semantic_only_model,
        train_loader,
        val_loader,
    )
)

Epoch 01 | loss=1.5606 | val_macro_f1=0.3250
Epoch 02 | loss=1.4367 | val_macro_f1=0.4380
Epoch 03 | loss=1.2679 | val_macro_f1=0.4247
Epoch 04 | loss=1.1174 | val_macro_f1=0.4166
Epoch 05 | loss=1.0273 | val_macro_f1=0.4175
Epoch 06 | loss=1.0334 | val_macro_f1=0.4395
Epoch 07 | loss=0.9324 | val_macro_f1=0.4337
Epoch 08 | loss=0.8984 | val_macro_f1=0.4646
Epoch 09 | loss=0.9379 | val_macro_f1=0.4497
Epoch 10 | loss=0.8736 | val_macro_f1=0.4443
Epoch 11 | loss=0.8726 | val_macro_f1=0.4511
Epoch 12 | loss=0.8765 | val_macro_f1=0.4570
Epoch 13 | loss=0.8318 | val_macro_f1=0.4625
Early stopping.


In [51]:
semantic_result = evaluate_model(
    semantic_only_model,
    test_loader,
)

{
    k: v
    for k, v
    in semantic_result.items()
    if k not in {
        "y_true",
        "y_pred",
    }
}

{'accuracy': 0.3832752613240418,
 'balanced_accuracy': 0.48804204186355216,
 'macro_f1': 0.38775137239157154,
 'weighted_f1': 0.3454599217991778}

In [52]:
torch.manual_seed(42)
np.random.seed(42)

trajectory_only_model = (
    TrajectoryOnlyModel(
        num_roles=len(
            role_vocab
        ),
        num_tools=len(
            tool_vocab
        ),
    )
)

trajectory_only_model, trajectory_history = (
    train_model(
        trajectory_only_model,
        train_loader,
        val_loader,
    )
)

Epoch 01 | loss=1.5621 | val_macro_f1=0.4500
Epoch 02 | loss=1.3900 | val_macro_f1=0.3710
Epoch 03 | loss=1.2316 | val_macro_f1=0.4282
Epoch 04 | loss=1.2228 | val_macro_f1=0.4302
Epoch 05 | loss=1.1497 | val_macro_f1=0.4574
Epoch 06 | loss=1.1387 | val_macro_f1=0.4604
Epoch 07 | loss=1.0966 | val_macro_f1=0.4575
Epoch 08 | loss=1.0608 | val_macro_f1=0.4320
Epoch 09 | loss=1.0333 | val_macro_f1=0.5278
Epoch 10 | loss=1.0472 | val_macro_f1=0.4034
Epoch 11 | loss=1.0410 | val_macro_f1=0.4250
Epoch 12 | loss=0.9720 | val_macro_f1=0.3942
Epoch 13 | loss=0.9539 | val_macro_f1=0.4278
Epoch 14 | loss=0.9254 | val_macro_f1=0.4024
Early stopping.


In [53]:
trajectory_result = evaluate_model(
    trajectory_only_model,
    test_loader,
)

{
    k: v
    for k, v
    in trajectory_result.items()
    if k not in {
        "y_true",
        "y_pred",
    }
}

{'accuracy': 0.30662020905923343,
 'balanced_accuracy': 0.4377952787106105,
 'macro_f1': 0.35662273978088727,
 'weighted_f1': 0.2528925735786182}

In [57]:
torch.manual_seed(42)
np.random.seed(42)

fusion_model = (
    SemanticTrajectoryModel(
        num_roles=len(
            role_vocab
        ),
        num_tools=len(
            tool_vocab
        ),
    )
)

fusion_model, fusion_history = (
    train_model(
        fusion_model,
        train_loader,
        val_loader,
    )
)

Epoch 01 | loss=1.5444 | val_macro_f1=0.3528
Epoch 02 | loss=1.2741 | val_macro_f1=0.2767
Epoch 03 | loss=1.1179 | val_macro_f1=0.4740
Epoch 04 | loss=1.0418 | val_macro_f1=0.4530
Epoch 05 | loss=0.9638 | val_macro_f1=0.5242
Epoch 06 | loss=0.9296 | val_macro_f1=0.4215
Epoch 07 | loss=0.8775 | val_macro_f1=0.4989
Epoch 08 | loss=0.8388 | val_macro_f1=0.5226
Epoch 09 | loss=0.7986 | val_macro_f1=0.5166
Epoch 10 | loss=0.7508 | val_macro_f1=0.4632
Early stopping.


In [58]:
fusion_result = evaluate_model(
    fusion_model,
    test_loader,
)

{
    k: v
    for k, v
    in fusion_result.items()
    if k not in {
        "y_true",
        "y_pred",
    }
}

{'accuracy': 0.36585365853658536,
 'balanced_accuracy': 0.4409531734474526,
 'macro_f1': 0.3921204959683851,
 'weighted_f1': 0.36037570416259807}

In [59]:
results = pd.DataFrame([
    {
        "model":
            "semantic_only",

        **{
            k: semantic_result[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },

    {
        "model":
            "trajectory_gru_only",

        **{
            k: trajectory_result[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },

    {
        "model":
            "semantic_plus_trajectory_gru",

        **{
            k: fusion_result[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },
])

results

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_only,0.383275,0.488042,0.387751,0.345460
1,trajectory_gru_only,0.306620,0.437795,0.356623,0.252893
2,semantic_plus_trajectory_gru,0.365854,0.440953,0.392120,0.360376


In [60]:
semantic_row = (
    results
    .set_index("model")
    .loc["semantic_only"]
)

for metric in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:

    results[
        f"delta_{metric}_vs_semantic"
    ] = (
        results[metric]
        - semantic_row[metric]
    )

results.sort_values(
    "macro_f1",
    ascending=False,
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_semantic,delta_balanced_accuracy_vs_semantic,delta_macro_f1_vs_semantic,delta_weighted_f1_vs_semantic
2,semantic_plus_trajectory_gru,0.365854,0.440953,0.392120,0.360376,-0.017422,-0.047089,0.004369,0.014916
0,semantic_only,0.383275,0.488042,0.387751,0.345460,0.000000,0.000000,0.000000,0.000000
1,trajectory_gru_only,0.306620,0.437795,0.356623,0.252893,-0.076655,-0.050247,-0.031129,-0.092567


In [61]:
family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

In [62]:
print("=" * 80)
print("SEMANTIC ONLY")
print("=" * 80)

print(
    classification_report(
        semantic_result["y_true"],
        semantic_result["y_pred"],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)


print("=" * 80)
print("TRAJECTORY GRU ONLY")
print("=" * 80)

print(
    classification_report(
        trajectory_result["y_true"],
        trajectory_result["y_pred"],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)


print("=" * 80)
print("SEMANTIC + TRAJECTORY GRU")
print("=" * 80)

print(
    classification_report(
        fusion_result["y_true"],
        fusion_result["y_pred"],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

SEMANTIC ONLY
                       precision    recall  f1-score   support

       workflow_error     0.5588    0.1377    0.2209       138
     constraint_error     0.4954    0.7714    0.6034        70
       tool_use_error     0.2055    0.3947    0.2703        38
grounding_state_error     0.2830    0.5000    0.3614        30
reasoning_value_error     0.3889    0.6364    0.4828        11

             accuracy                         0.3833       287
            macro avg     0.3863    0.4880    0.3878       287
         weighted avg     0.4612    0.3833    0.3455       287

TRAJECTORY GRU ONLY
                       precision    recall  f1-score   support

       workflow_error     0.3077    0.0290    0.0530       138
     constraint_error     0.4731    0.6286    0.5399        70
       tool_use_error     0.3088    0.5526    0.3962        38
grounding_state_error     0.1250    0.4333    0.1940        30
reasoning_value_error     0.6667    0.5455    0.6000        11

             acc

In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

y_train_full = (
    train_targets["family_label"]
    .to_numpy()
)

y_test_full = (
    test_targets["family_label"]
    .to_numpy()
)

semantic_lr = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=42,
)

semantic_lr.fit(
    X_semantic_train,
    y_train_full,
)

semantic_lr_pred = semantic_lr.predict(
    X_semantic_test
)

semantic_lr_result = {
    "accuracy": accuracy_score(
        y_test_full,
        semantic_lr_pred,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_test_full,
        semantic_lr_pred,
    ),
    "macro_f1": f1_score(
        y_test_full,
        semantic_lr_pred,
        average="macro",
        zero_division=0,
    ),
    "weighted_f1": f1_score(
        y_test_full,
        semantic_lr_pred,
        average="weighted",
        zero_division=0,
    ),
}

print(semantic_lr_result)

print(
    classification_report(
        y_test_full,
        semantic_lr_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

{'accuracy': 0.37282229965156793, 'balanced_accuracy': 0.5131896936015929, 'macro_f1': 0.38912129811616375, 'weighted_f1': 0.31923286550285296}
                       precision    recall  f1-score   support

       workflow_error     0.5000    0.0797    0.1375       138
     constraint_error     0.5591    0.7429    0.6380        70
       tool_use_error     0.2169    0.4737    0.2975        38
grounding_state_error     0.2639    0.6333    0.3725        30
reasoning_value_error     0.4118    0.6364    0.5000        11

             accuracy                         0.3728       287
            macro avg     0.3903    0.5132    0.3891       287
         weighted avg     0.4489    0.3728    0.3192       287



In [64]:
old_train = pd.read_csv(
    "../data/processed/taxonomy_train.csv"
)

old_test = pd.read_csv(
    "../data/processed/taxonomy_test.csv"
)

key_cols = [
    "dataset",
    "group_id",
    "message_index",
]

train_check = train_targets.merge(
    old_train[
        key_cols + ["current_text"]
    ],
    on=key_cols,
    how="left",
    validate="one_to_one",
)

test_check = test_targets.merge(
    old_test[
        key_cols + ["current_text"]
    ],
    on=key_cols,
    how="left",
    validate="one_to_one",
)

print(
    "Train exact text equality:",
    (
        train_check["content"].fillna("")
        ==
        train_check["current_text"].fillna("")
    ).mean()
)

print(
    "Test exact text equality:",
    (
        test_check["content"].fillna("")
        ==
        test_check["current_text"].fillna("")
    ).mean()
)

Train exact text equality: 1.0
Test exact text equality: 1.0


In [65]:
# ============================================================
# REPRODUCE CANONICAL SEMANTIC LR BASELINE
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

y_train_full = (
    train_targets["family_label"]
    .to_numpy()
)

y_test_full = (
    test_targets["family_label"]
    .to_numpy()
)


semantic_lr_unweighted = LogisticRegression(
    max_iter=5000,
    random_state=42,
)

semantic_lr_unweighted.fit(
    X_semantic_train,
    y_train_full,
)

semantic_lr_unweighted_pred = (
    semantic_lr_unweighted.predict(
        X_semantic_test
    )
)

semantic_lr_unweighted_result = {
    "accuracy":
        accuracy_score(
            y_test_full,
            semantic_lr_unweighted_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test_full,
            semantic_lr_unweighted_pred,
        ),

    "macro_f1":
        f1_score(
            y_test_full,
            semantic_lr_unweighted_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test_full,
            semantic_lr_unweighted_pred,
            average="weighted",
            zero_division=0,
        ),
}

print(
    semantic_lr_unweighted_result
)

print(
    classification_report(
        y_test_full,
        semantic_lr_unweighted_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

{'accuracy': 0.5156794425087108, 'balanced_accuracy': 0.4813912250983189, 'macro_f1': 0.4902679078131794, 'weighted_f1': 0.5214873265322301}
                       precision    recall  f1-score   support

       workflow_error     0.6230    0.5507    0.5846       138
     constraint_error     0.5584    0.6143    0.5850        70
       tool_use_error     0.3030    0.2632    0.2817        38
grounding_state_error     0.2708    0.4333    0.3333        30
reasoning_value_error     0.8571    0.5455    0.6667        11

             accuracy                         0.5157       287
            macro avg     0.5225    0.4814    0.4903       287
         weighted avg     0.5370    0.5157    0.5215       287



In [66]:
semantic_weight_comparison = pd.DataFrame([
    {
        "model": "LR_balanced",
        "accuracy": 0.37282229965156793,
        "balanced_accuracy": 0.5131896936015929,
        "macro_f1": 0.38912129811616375,
        "weighted_f1": 0.31923286550285296,
    },
    {
        "model": "LR_unweighted",
        **semantic_lr_unweighted_result,
    },
])

semantic_weight_comparison

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,LR_balanced,0.372822,0.513190,0.389121,0.319233
1,LR_unweighted,0.515679,0.481391,0.490268,0.521487


In [68]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

In [69]:
def train_model(
    model,
    train_loader,
    val_loader,
    epochs=30,
    lr=1e-3,
    patience=5,
    use_class_weights=False,
):

    model = model.to(device)

    if use_class_weights:
        criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(device)
        )
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    best_state = None
    best_macro_f1 = -np.inf
    patience_counter = 0
    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        losses = []

        for batch in train_loader:

            batch = move_batch(
                batch,
                device,
            )

            optimizer.zero_grad()

            logits = model(batch)

            loss = criterion(
                logits,
                batch["labels"],
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            losses.append(
                loss.item()
            )

        val_metrics = evaluate_model(
            model,
            val_loader,
        )

        row = {
            "epoch": epoch,
            "train_loss":
                np.mean(losses),

            "val_accuracy":
                val_metrics["accuracy"],

            "val_balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ],

            "val_macro_f1":
                val_metrics["macro_f1"],

            "val_weighted_f1":
                val_metrics[
                    "weighted_f1"
                ],
        }

        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"loss={row['train_loss']:.4f} | "
            f"val_macro_f1="
            f"{row['val_macro_f1']:.4f}"
        )

        if (
            row["val_macro_f1"]
            > best_macro_f1
        ):

            best_macro_f1 = (
                row["val_macro_f1"]
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= patience:

            print("Early stopping.")
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(history),
    )

In [70]:
torch.manual_seed(42)
np.random.seed(42)

semantic_only_model = (
    SemanticOnlyModel()
)

semantic_only_model, semantic_history = (
    train_model(
        semantic_only_model,
        train_loader,
        val_loader,
        use_class_weights=False,
    )
)

semantic_result = evaluate_model(
    semantic_only_model,
    test_loader,
)

{
    k: v
    for k, v in semantic_result.items()
    if k not in {"y_true", "y_pred"}
}

Epoch 01 | loss=1.4442 | val_macro_f1=0.1279
Epoch 02 | loss=1.2705 | val_macro_f1=0.1897
Epoch 03 | loss=1.1533 | val_macro_f1=0.2114
Epoch 04 | loss=1.0884 | val_macro_f1=0.2987
Epoch 05 | loss=1.0401 | val_macro_f1=0.3285
Epoch 06 | loss=1.0332 | val_macro_f1=0.3168
Epoch 07 | loss=0.9554 | val_macro_f1=0.4963
Epoch 08 | loss=0.9334 | val_macro_f1=0.4892
Epoch 09 | loss=0.9546 | val_macro_f1=0.4866
Epoch 10 | loss=0.9023 | val_macro_f1=0.4987
Epoch 11 | loss=0.8912 | val_macro_f1=0.4893
Epoch 12 | loss=0.8987 | val_macro_f1=0.5069
Epoch 13 | loss=0.8583 | val_macro_f1=0.4561
Epoch 14 | loss=0.8689 | val_macro_f1=0.4678
Epoch 15 | loss=0.8295 | val_macro_f1=0.4549
Epoch 16 | loss=0.8019 | val_macro_f1=0.4713
Epoch 17 | loss=0.7869 | val_macro_f1=0.4619
Early stopping.


{'accuracy': 0.49825783972125437,
 'balanced_accuracy': 0.4875827909695186,
 'macro_f1': 0.4904623953756964,
 'weighted_f1': 0.5088361096953921}

In [71]:
torch.manual_seed(42)
np.random.seed(42)

trajectory_only_model = (
    TrajectoryOnlyModel(
        num_roles=len(role_vocab),
        num_tools=len(tool_vocab),
    )
)

trajectory_only_model, trajectory_history = (
    train_model(
        trajectory_only_model,
        train_loader,
        val_loader,
        use_class_weights=False,
    )
)

trajectory_result = evaluate_model(
    trajectory_only_model,
    test_loader,
)

Epoch 01 | loss=1.4026 | val_macro_f1=0.1236
Epoch 02 | loss=1.2374 | val_macro_f1=0.2225
Epoch 03 | loss=1.1766 | val_macro_f1=0.3118
Epoch 04 | loss=1.1423 | val_macro_f1=0.4349
Epoch 05 | loss=1.0781 | val_macro_f1=0.3620
Epoch 06 | loss=1.0866 | val_macro_f1=0.3884
Epoch 07 | loss=1.0420 | val_macro_f1=0.4074
Epoch 08 | loss=1.0069 | val_macro_f1=0.3780
Epoch 09 | loss=0.9957 | val_macro_f1=0.4260
Early stopping.


In [72]:
torch.manual_seed(42)
np.random.seed(42)

fusion_model = (
    SemanticTrajectoryModel(
        num_roles=len(role_vocab),
        num_tools=len(tool_vocab),
    )
)

fusion_model, fusion_history = (
    train_model(
        fusion_model,
        train_loader,
        val_loader,
        use_class_weights=False,
    )
)

fusion_result = evaluate_model(
    fusion_model,
    test_loader,
)

Epoch 01 | loss=1.3916 | val_macro_f1=0.1538
Epoch 02 | loss=1.1935 | val_macro_f1=0.3185
Epoch 03 | loss=1.0822 | val_macro_f1=0.4261
Epoch 04 | loss=1.0207 | val_macro_f1=0.5042
Epoch 05 | loss=0.9936 | val_macro_f1=0.4779
Epoch 06 | loss=0.9212 | val_macro_f1=0.4047
Epoch 07 | loss=0.8731 | val_macro_f1=0.4872
Epoch 08 | loss=0.8707 | val_macro_f1=0.4571
Epoch 09 | loss=0.8170 | val_macro_f1=0.4948
Early stopping.


In [73]:
trajectory_result = evaluate_model(
    trajectory_only_model,
    test_loader,
)

fusion_result = evaluate_model(
    fusion_model,
    test_loader,
)

print("TRAJECTORY ONLY")
print({
    k: v
    for k, v in trajectory_result.items()
    if k not in {"y_true", "y_pred"}
})

print("\nSEMANTIC + TRAJECTORY")
print({
    k: v
    for k, v in fusion_result.items()
    if k not in {"y_true", "y_pred"}
})

TRAJECTORY ONLY
{'accuracy': 0.49477351916376305, 'balanced_accuracy': 0.4006476665973233, 'macro_f1': 0.4166407318350059, 'weighted_f1': 0.4905422238221166}

SEMANTIC + TRAJECTORY
{'accuracy': 0.42857142857142855, 'balanced_accuracy': 0.38652758378158836, 'macro_f1': 0.3888515869505697, 'weighted_f1': 0.43437806393417444}


In [74]:
results = pd.DataFrame([
    {
        "model": "semantic_only",
        **{
            k: semantic_result[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },
    {
        "model": "trajectory_gru_only",
        **{
            k: trajectory_result[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },
    {
        "model": "semantic_plus_trajectory_gru",
        **{
            k: fusion_result[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },
])

baseline = (
    results
    .set_index("model")
    .loc["semantic_only"]
)

for metric in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:
    results[
        f"delta_{metric}_vs_semantic"
    ] = (
        results[metric]
        - baseline[metric]
    )

display(
    results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_semantic,delta_balanced_accuracy_vs_semantic,delta_macro_f1_vs_semantic,delta_weighted_f1_vs_semantic
0,semantic_only,0.4983,0.4876,0.4905,0.5088,0.0000,0.0000,0.0000,0.0000
1,trajectory_gru_only,0.4948,0.4006,0.4166,0.4905,-0.0035,-0.0869,-0.0738,-0.0183
2,semantic_plus_trajectory_gru,0.4286,0.3865,0.3889,0.4344,-0.0697,-0.1011,-0.1016,-0.0745


In [75]:
from sklearn.metrics import classification_report

print("=" * 80)
print("TRAJECTORY GRU ONLY")
print("=" * 80)

print(
    classification_report(
        trajectory_result["y_true"],
        trajectory_result["y_pred"],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

print("=" * 80)
print("SEMANTIC + TRAJECTORY GRU")
print("=" * 80)

print(
    classification_report(
        fusion_result["y_true"],
        fusion_result["y_pred"],
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

TRAJECTORY GRU ONLY
                       precision    recall  f1-score   support

       workflow_error     0.5902    0.5217    0.5538       138
     constraint_error     0.5269    0.7000    0.6012        70
       tool_use_error     0.3023    0.3421    0.3210        38
grounding_state_error     0.1923    0.1667    0.1786        30
reasoning_value_error     1.0000    0.2727    0.4286        11

             accuracy                         0.4948       287
            macro avg     0.5223    0.4006    0.4166       287
         weighted avg     0.5107    0.4948    0.4905       287

SEMANTIC + TRAJECTORY GRU
                       precision    recall  f1-score   support

       workflow_error     0.5089    0.4130    0.4560       138
     constraint_error     0.5195    0.5714    0.5442        70
       tool_use_error     0.2766    0.3421    0.3059        38
grounding_state_error     0.2174    0.3333    0.2632        30
reasoning_value_error     0.6000    0.2727    0.3750        11

    

In [76]:
def encode_trajectory(self, batch):

    # Use the SAME trajectory encoding logic
    # currently used inside forward().

    role_emb = self.role_embedding(
        batch["roles"]
    )

    tool_emb = self.tool_embedding(
        batch["tools"]
    )

    x = torch.cat(
        [
            role_emb,
            tool_emb,
            batch["event_features"],
        ],
        dim=-1,
    )

    lengths = batch["lengths"]

    packed = torch.nn.utils.rnn.pack_padded_sequence(
        x,
        lengths.cpu(),
        batch_first=True,
        enforce_sorted=False,
    )

    _, hidden = self.gru(packed)

    trajectory_embedding = hidden[-1]

    return trajectory_embedding

In [81]:
def extract_trajectory_embeddings(
    model,
    loader,
    device,
):
    model.eval()

    embeddings = []
    labels = []

    with torch.no_grad():

        for batch in loader:

            batch = move_batch(
                batch,
                device,
            )

            # Use the existing encoder directly
            h = model.encoder(
                batch["role_ids"],
                batch["tool_ids"],
                batch["numeric"],
                batch["lengths"],
            )

            embeddings.append(
                h.cpu().numpy()
            )

            labels.append(
                batch["labels"]
                .cpu()
                .numpy()
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        np.concatenate(
            labels,
            axis=0,
        ),
    )

In [89]:
from torch.utils.data import DataLoader

# ============================================================
# FULL CANONICAL TRAIN DATASET FOR REPRESENTATION EXTRACTION
# ============================================================

full_train_dataset = TrajectoryDataset(
    train_targets,
    train_event_lookup,
    X_semantic_train,
    role_vocab,
    tool_vocab,
    numeric_mean,
    numeric_std,
)

full_train_loader_no_shuffle = DataLoader(
    full_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=trajectory_collate,
)

print(
    "Full train dataset:",
    len(full_train_dataset)
)

assert len(full_train_dataset) == 1489

train_loader_no_shuffle = DataLoader(
    train_loader.dataset,
    batch_size=train_loader.batch_size,
    shuffle=False,
    collate_fn=train_loader.collate_fn,
    num_workers=getattr(train_loader, "num_workers", 0),
)

print("Train dataset:", len(train_loader_no_shuffle.dataset))
print("Batch size:", train_loader_no_shuffle.batch_size)
print("Shuffle: False")

Full train dataset: 1489
Train dataset: 1185
Batch size: 32
Shuffle: False


In [90]:
H_traj_train, y_traj_train = (
    extract_trajectory_embeddings(
        trajectory_only_model,
        train_loader_no_shuffle,
        device,
    )
)

H_traj_test, y_traj_test = (
    extract_trajectory_embeddings(
        trajectory_only_model,
        test_loader,
        device,
    )
)

print(
    "Train trajectory representation:",
    H_traj_train.shape
)

print(
    "Test trajectory representation:",
    H_traj_test.shape
)

Train trajectory representation: (1185, 128)
Test trajectory representation: (287, 128)


In [92]:
H_traj_train, y_traj_train = (
    extract_trajectory_embeddings(
        trajectory_only_model,
        full_train_loader_no_shuffle,
        device,
    )
)

H_traj_test, y_traj_test = (
    extract_trajectory_embeddings(
        trajectory_only_model,
        test_loader,
        device,
    )
)

print(
    "Train trajectory representation:",
    H_traj_train.shape
)

print(
    "Test trajectory representation:",
    H_traj_test.shape
)

print(
    "Train labels:",
    y_traj_train.shape
)

print(
    "Test labels:",
    y_traj_test.shape
)

Train trajectory representation: (1489, 128)
Test trajectory representation: (287, 128)
Train labels: (1489,)
Test labels: (287,)


In [93]:
np.testing.assert_array_equal(
    y_traj_train.astype(int),
    train_targets[
        "family_label"
    ].to_numpy().astype(int),
)

np.testing.assert_array_equal(
    y_traj_test.astype(int),
    test_targets[
        "family_label"
    ].to_numpy().astype(int),
)

print(
    "✓ Trajectory embeddings aligned "
    "with canonical train/test targets"
)

✓ Trajectory embeddings aligned with canonical train/test targets


In [94]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Prepare labels
# ---------------------------------------------------------

y_train_probe = y_traj_train.astype(int)
y_test_probe = y_traj_test.astype(int)

# ---------------------------------------------------------
# 2. Semantic-only probe
# ---------------------------------------------------------

lr_semantic = LogisticRegression(
    max_iter=5000,
    random_state=42,
)

lr_semantic.fit(
    X_semantic_train,
    y_train_probe,
)

pred_semantic_probe = lr_semantic.predict(
    X_semantic_test
)

# ---------------------------------------------------------
# 3. Trajectory-only probe
# ---------------------------------------------------------

lr_trajectory = LogisticRegression(
    max_iter=5000,
    random_state=42,
)

lr_trajectory.fit(
    H_traj_train,
    y_train_probe,
)

pred_trajectory_probe = lr_trajectory.predict(
    H_traj_test
)

# ---------------------------------------------------------
# 4. Semantic + trajectory probe
# ---------------------------------------------------------

X_probe_fused_train = np.hstack([
    X_semantic_train,
    H_traj_train,
])

X_probe_fused_test = np.hstack([
    X_semantic_test,
    H_traj_test,
])

lr_fused = LogisticRegression(
    max_iter=5000,
    random_state=42,
)

lr_fused.fit(
    X_probe_fused_train,
    y_train_probe,
)

pred_fused_probe = lr_fused.predict(
    X_probe_fused_test
)

In [95]:
def metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

probe_results = pd.DataFrame([
    {
        "model": "semantic_embedding_lr",
        **metrics(
            y_test_probe,
            pred_semantic_probe,
        ),
    },
    {
        "model": "trajectory_embedding_lr",
        **metrics(
            y_test_probe,
            pred_trajectory_probe,
        ),
    },
    {
        "model": "semantic_plus_trajectory_lr",
        **metrics(
            y_test_probe,
            pred_fused_probe,
        ),
    },
])

display(
    probe_results.sort_values(
        "macro_f1",
        ascending=False,
    ).round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic_embedding_lr,0.5157,0.4814,0.4903,0.5215
1,trajectory_embedding_lr,0.4774,0.4851,0.4700,0.4856
2,semantic_plus_trajectory_lr,0.4425,0.4431,0.4430,0.4556


In [96]:
baseline = (
    probe_results
    .set_index("model")
    .loc["semantic_embedding_lr"]
)

for metric_name in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:
    probe_results[
        f"delta_{metric_name}_vs_semantic"
    ] = (
        probe_results[metric_name]
        - baseline[metric_name]
    )

display(
    probe_results
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_semantic,delta_balanced_accuracy_vs_semantic,delta_macro_f1_vs_semantic,delta_weighted_f1_vs_semantic
0,semantic_embedding_lr,0.5157,0.4814,0.4903,0.5215,0.0000,0.0000,0.0000,0.0000
1,trajectory_embedding_lr,0.4774,0.4851,0.4700,0.4856,-0.0383,0.0037,-0.0202,-0.0359
2,semantic_plus_trajectory_lr,0.4425,0.4431,0.4430,0.4556,-0.0732,-0.0383,-0.0473,-0.0658


In [97]:
family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

for name, pred in [
    (
        "SEMANTIC EMBEDDING LR",
        pred_semantic_probe,
    ),
    (
        "TRAJECTORY EMBEDDING LR",
        pred_trajectory_probe,
    ),
    (
        "SEMANTIC + TRAJECTORY LR",
        pred_fused_probe,
    ),
]:
    print("=" * 80)
    print(name)
    print("=" * 80)

    print(
        classification_report(
            y_test_probe,
            pred,
            target_names=family_names,
            digits=4,
            zero_division=0,
        )
    )

SEMANTIC EMBEDDING LR
                       precision    recall  f1-score   support

       workflow_error     0.6230    0.5507    0.5846       138
     constraint_error     0.5584    0.6143    0.5850        70
       tool_use_error     0.3030    0.2632    0.2817        38
grounding_state_error     0.2708    0.4333    0.3333        30
reasoning_value_error     0.8571    0.5455    0.6667        11

             accuracy                         0.5157       287
            macro avg     0.5225    0.4814    0.4903       287
         weighted avg     0.5370    0.5157    0.5215       287

TRAJECTORY EMBEDDING LR
                       precision    recall  f1-score   support

       workflow_error     0.6061    0.4348    0.5063       138
     constraint_error     0.5325    0.5857    0.5578        70
       tool_use_error     0.3333    0.5263    0.4082        38
grounding_state_error     0.2381    0.3333    0.2778        30
reasoning_value_error     0.6667    0.5455    0.6000        11

    

In [98]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ============================================================
# 1. Learn trajectory component predictable from semantics
# ============================================================

trajectory_from_semantic = Ridge(
    alpha=10.0
)

trajectory_from_semantic.fit(
    X_semantic_train,
    H_traj_train,
)

H_traj_train_pred = trajectory_from_semantic.predict(
    X_semantic_train
)

H_traj_test_pred = trajectory_from_semantic.predict(
    X_semantic_test
)

# ------------------------------------------------------------
# Residual trajectory representation
# ------------------------------------------------------------

H_traj_train_residual = (
    H_traj_train
    - H_traj_train_pred
)

H_traj_test_residual = (
    H_traj_test
    - H_traj_test_pred
)

print(
    "Residual train:",
    H_traj_train_residual.shape
)

print(
    "Residual test:",
    H_traj_test_residual.shape
)

Residual train: (1489, 128)
Residual test: (287, 128)


In [99]:
lr_traj_residual = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        random_state=42,
    ),
)

lr_traj_residual.fit(
    H_traj_train_residual,
    y_train_probe,
)

pred_traj_residual = (
    lr_traj_residual.predict(
        H_traj_test_residual
    )
)

residual_only_results = metrics(
    y_test_probe,
    pred_traj_residual,
)

print(
    "Trajectory residual only:",
    residual_only_results
)

Trajectory residual only: {'accuracy': 0.3832752613240418, 'balanced_accuracy': 0.305308726361358, 'macro_f1': 0.33182404845030444, 'weighted_f1': 0.39347771694942485}


In [100]:
X_residual_fused_train = np.hstack([
    X_semantic_train,
    H_traj_train_residual,
])

X_residual_fused_test = np.hstack([
    X_semantic_test,
    H_traj_test_residual,
])

lr_residual_fused = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        random_state=42,
    ),
)

lr_residual_fused.fit(
    X_residual_fused_train,
    y_train_probe,
)

pred_residual_fused = (
    lr_residual_fused.predict(
        X_residual_fused_test
    )
)

residual_fused_results = metrics(
    y_test_probe,
    pred_residual_fused,
)

print(
    "Semantic + residual trajectory:",
    residual_fused_results
)

Semantic + residual trajectory: {'accuracy': 0.43902439024390244, 'balanced_accuracy': 0.44955015998494263, 'macro_f1': 0.43935304868736, 'weighted_f1': 0.4504156430194012}


In [101]:
probe_results_extended = pd.DataFrame([
    {
        "model": "semantic",
        **metrics(
            y_test_probe,
            pred_semantic_probe,
        ),
    },
    {
        "model": "trajectory",
        **metrics(
            y_test_probe,
            pred_trajectory_probe,
        ),
    },
    {
        "model": "semantic_plus_trajectory",
        **metrics(
            y_test_probe,
            pred_fused_probe,
        ),
    },
    {
        "model": "trajectory_residual",
        **residual_only_results,
    },
    {
        "model": "semantic_plus_residual_trajectory",
        **residual_fused_results,
    },
])

baseline = (
    probe_results_extended
    .set_index("model")
    .loc["semantic"]
)

for metric_name in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:
    probe_results_extended[
        f"delta_{metric_name}_vs_semantic"
    ] = (
        probe_results_extended[metric_name]
        - baseline[metric_name]
    )

display(
    probe_results_extended
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy_vs_semantic,delta_balanced_accuracy_vs_semantic,delta_macro_f1_vs_semantic,delta_weighted_f1_vs_semantic
0,semantic,0.5157,0.4814,0.4903,0.5215,0.0000,0.0000,0.0000,0.0000
1,trajectory,0.4774,0.4851,0.4700,0.4856,-0.0383,0.0037,-0.0202,-0.0359
2,semantic_plus_trajectory,0.4425,0.4431,0.4430,0.4556,-0.0732,-0.0383,-0.0473,-0.0658
4,semantic_plus_residual_trajectory,0.4390,0.4496,0.4394,0.4504,-0.0767,-0.0318,-0.0509,-0.0711
3,trajectory_residual,0.3833,0.3053,0.3318,0.3935,-0.1324,-0.1761,-0.1584,-0.1280


In [102]:
print(
    classification_report(
        y_test_probe,
        pred_residual_fused,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

                       precision    recall  f1-score   support

       workflow_error     0.5872    0.4638    0.5182       138
     constraint_error     0.3766    0.4143    0.3946        70
       tool_use_error     0.4318    0.5000    0.4634        38
grounding_state_error     0.1522    0.2333    0.1842        30
reasoning_value_error     0.6364    0.6364    0.6364        11

             accuracy                         0.4390       287
            macro avg     0.4368    0.4496    0.4394       287
         weighted avg     0.4717    0.4390    0.4504       287



Can trajectory history learn a selective correction to an already-good semantic classifier, rather than participating equally in every prediction?

In [103]:
from sklearn.linear_model import LogisticRegression

y_all_train = (
    train_targets["family_label"]
    .to_numpy()
    .astype(int)
)

y_test_final = (
    test_targets["family_label"]
    .to_numpy()
    .astype(int)
)

# ---------------------------------------------------------
# Semantic anchor
# ---------------------------------------------------------

semantic_anchor = LogisticRegression(
    max_iter=5000,
    random_state=42,
)

semantic_anchor.fit(
    X_semantic_train[inner_train_idx],
    y_all_train[inner_train_idx],
)

# Decision-function logits
semantic_logits_inner = semantic_anchor.decision_function(
    X_semantic_train[inner_train_idx]
)

semantic_logits_val = semantic_anchor.decision_function(
    X_semantic_train[val_idx]
)

semantic_logits_test = semantic_anchor.decision_function(
    X_semantic_test
)

print(semantic_logits_inner.shape)
print(semantic_logits_val.shape)
print(semantic_logits_test.shape)

(1185, 5)
(304, 5)
(287, 5)


In [104]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

anchor_test_pred = (
    semantic_logits_test.argmax(axis=1)
)

anchor_results = {
    "accuracy":
        accuracy_score(
            y_test_final,
            anchor_test_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test_final,
            anchor_test_pred,
        ),

    "macro_f1":
        f1_score(
            y_test_final,
            anchor_test_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test_final,
            anchor_test_pred,
            average="weighted",
            zero_division=0,
        ),
}

anchor_results

{'accuracy': 0.5296167247386759,
 'balanced_accuracy': 0.4610926525800668,
 'macro_f1': 0.4822937481503712,
 'weighted_f1': 0.527807366755815}

In [105]:
print(
    "Trajectory:",
    H_traj_train.shape,
    H_traj_test.shape,
)

print(
    "Semantic:",
    X_semantic_train.shape,
    X_semantic_test.shape,
)

assert len(H_traj_train) == 1489
assert len(H_traj_test) == 287

Trajectory: (1489, 128) (287, 128)
Semantic: (1489, 384) (287, 384)


In [106]:
H_traj_inner = (
    H_traj_train[
        inner_train_idx
    ]
)

H_traj_val = (
    H_traj_train[
        val_idx
    ]
)

print(
    H_traj_inner.shape,
    H_traj_val.shape,
    H_traj_test.shape,
)

(1185, 128) (304, 128) (287, 128)


In [107]:
import torch
from torch.utils.data import (
    Dataset,
    DataLoader,
)

In [108]:
class ResidualCorrectionDataset(Dataset):

    def __init__(
        self,
        semantic_logits,
        trajectory_embeddings,
        labels,
    ):

        self.semantic_logits = torch.tensor(
            semantic_logits,
            dtype=torch.float32,
        )

        self.trajectory = torch.tensor(
            trajectory_embeddings,
            dtype=torch.float32,
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return {
            "semantic_logits":
                self.semantic_logits[idx],

            "trajectory":
                self.trajectory[idx],

            "labels":
                self.labels[idx],
        }

In [109]:
residual_train_dataset = (
    ResidualCorrectionDataset(
        semantic_logits_inner,
        H_traj_inner,
        y_all_train[
            inner_train_idx
        ],
    )
)

residual_val_dataset = (
    ResidualCorrectionDataset(
        semantic_logits_val,
        H_traj_val,
        y_all_train[
            val_idx
        ],
    )
)

residual_test_dataset = (
    ResidualCorrectionDataset(
        semantic_logits_test,
        H_traj_test,
        y_test_final,
    )
)

In [110]:
RESIDUAL_BATCH_SIZE = 32

residual_train_loader = DataLoader(
    residual_train_dataset,
    batch_size=RESIDUAL_BATCH_SIZE,
    shuffle=True,
)

residual_val_loader = DataLoader(
    residual_val_dataset,
    batch_size=RESIDUAL_BATCH_SIZE,
    shuffle=False,
)

residual_test_loader = DataLoader(
    residual_test_dataset,
    batch_size=RESIDUAL_BATCH_SIZE,
    shuffle=False,
)

In [111]:
class GatedTrajectoryCorrection(nn.Module):

    def __init__(
        self,
        trajectory_dim,
        num_classes=5,
        hidden_dim=64,
        gate_bias=-3.0,
    ):

        super().__init__()

        # ---------------------------------------------------
        # Trajectory correction network
        # ---------------------------------------------------

        self.correction = nn.Sequential(
            nn.Linear(
                trajectory_dim,
                hidden_dim,
            ),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(
                hidden_dim,
                num_classes,
            ),
        )

        # ---------------------------------------------------
        # Gate
        #
        # Input:
        # semantic logits + trajectory representation
        # ---------------------------------------------------

        self.gate_hidden = nn.Sequential(
            nn.Linear(
                trajectory_dim
                + num_classes,
                32,
            ),
            nn.ReLU(),
            nn.Dropout(0.10),
        )

        self.gate_output = nn.Linear(
            32,
            1,
        )

        # Start near semantic-only behavior.
        nn.init.constant_(
            self.gate_output.bias,
            gate_bias,
        )

        # Start correction itself near zero.
        nn.init.zeros_(
            self.correction[-1].weight
        )

        nn.init.zeros_(
            self.correction[-1].bias
        )

    def forward(
        self,
        semantic_logits,
        trajectory,
    ):

        delta_logits = (
            self.correction(
                trajectory
            )
        )

        gate_features = torch.cat(
            [
                semantic_logits,
                trajectory,
            ],
            dim=1,
        )

        gate_hidden = (
            self.gate_hidden(
                gate_features
            )
        )

        alpha = torch.sigmoid(
            self.gate_output(
                gate_hidden
            )
        )

        final_logits = (
            semantic_logits
            +
            alpha
            * delta_logits
        )

        return {
            "logits":
                final_logits,

            "alpha":
                alpha.squeeze(1),

            "delta_logits":
                delta_logits,
        }

In [112]:
TRAJECTORY_DIM = (
    H_traj_train.shape[1]
)

residual_model = (
    GatedTrajectoryCorrection(
        trajectory_dim=TRAJECTORY_DIM,
        num_classes=5,
        hidden_dim=64,
        gate_bias=-3.0,
    )
)

print(residual_model)

GatedTrajectoryCorrection(
  (correction): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=5, bias=True)
  )
  (gate_hidden): Sequential(
    (0): Linear(in_features=133, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (gate_output): Linear(in_features=32, out_features=1, bias=True)
)


In [113]:
@torch.no_grad()
def evaluate_residual_model(
    model,
    loader,
):

    model.eval()

    y_true = []
    y_pred = []

    all_alpha = []
    all_delta_norm = []

    for batch in loader:

        semantic_logits = (
            batch[
                "semantic_logits"
            ]
            .to(device)
        )

        trajectory = (
            batch[
                "trajectory"
            ]
            .to(device)
        )

        labels = (
            batch["labels"]
            .to(device)
        )

        output = model(
            semantic_logits,
            trajectory,
        )

        pred = (
            output["logits"]
            .argmax(dim=1)
        )

        y_true.extend(
            labels.cpu().numpy()
        )

        y_pred.extend(
            pred.cpu().numpy()
        )

        all_alpha.extend(
            output["alpha"]
            .cpu()
            .numpy()
        )

        all_delta_norm.extend(
            output["delta_logits"]
            .norm(
                dim=1
            )
            .cpu()
            .numpy()
        )

    y_true = np.asarray(
        y_true
    )

    y_pred = np.asarray(
        y_pred
    )

    return {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "alpha":
            np.asarray(
                all_alpha
            ),

        "delta_norm":
            np.asarray(
                all_delta_norm
            ),
    }

In [114]:
def train_residual_model(
    model,
    train_loader,
    val_loader,
    epochs=40,
    lr=5e-4,
    patience=7,
    lambda_gate=0.01,
    lambda_delta=0.001,
):

    model = model.to(
        device
    )

    criterion = (
        nn.CrossEntropyLoss()
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    best_state = None
    best_macro_f1 = -np.inf

    patience_counter = 0
    history = []

    for epoch in range(
        1,
        epochs + 1,
    ):

        model.train()

        epoch_losses = []
        epoch_gates = []

        for batch in train_loader:

            semantic_logits = (
                batch[
                    "semantic_logits"
                ]
                .to(device)
            )

            trajectory = (
                batch[
                    "trajectory"
                ]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            optimizer.zero_grad()

            output = model(
                semantic_logits,
                trajectory,
            )

            classification_loss = (
                criterion(
                    output["logits"],
                    labels,
                )
            )

            gate_penalty = (
                output["alpha"]
                .mean()
            )

            delta_penalty = (
                output[
                    "delta_logits"
                ]
                .pow(2)
                .mean()
            )

            loss = (
                classification_loss
                + lambda_gate
                * gate_penalty
                + lambda_delta
                * delta_penalty
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            epoch_losses.append(
                loss.item()
            )

            epoch_gates.append(
                output[
                    "alpha"
                ]
                .detach()
                .mean()
                .item()
            )

        val_result = (
            evaluate_residual_model(
                model,
                val_loader,
            )
        )

        row = {
            "epoch":
                epoch,

            "train_loss":
                np.mean(
                    epoch_losses
                ),

            "train_mean_gate":
                np.mean(
                    epoch_gates
                ),

            "val_accuracy":
                val_result[
                    "accuracy"
                ],

            "val_balanced_accuracy":
                val_result[
                    "balanced_accuracy"
                ],

            "val_macro_f1":
                val_result[
                    "macro_f1"
                ],

            "val_weighted_f1":
                val_result[
                    "weighted_f1"
                ],

            "val_mean_gate":
                val_result[
                    "alpha"
                ].mean(),
        }

        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"loss={row['train_loss']:.4f} | "
            f"gate={row['train_mean_gate']:.3f} | "
            f"val_macro_f1="
            f"{row['val_macro_f1']:.4f}"
        )

        if (
            row["val_macro_f1"]
            > best_macro_f1
        ):

            best_macro_f1 = (
                row[
                    "val_macro_f1"
                ]
            )

            best_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            patience_counter = 0

        else:

            patience_counter += 1

        if (
            patience_counter
            >= patience
        ):

            print(
                "Early stopping."
            )

            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        pd.DataFrame(
            history
        ),
    )

In [115]:
torch.manual_seed(42)
np.random.seed(42)

residual_model = (
    GatedTrajectoryCorrection(
        trajectory_dim=TRAJECTORY_DIM,
        num_classes=5,
        hidden_dim=64,
        gate_bias=-3.0,
    )
)

residual_model, residual_history = (
    train_residual_model(
        residual_model,
        residual_train_loader,
        residual_val_loader,
        epochs=40,
        lr=5e-4,
        patience=7,
        lambda_gate=0.01,
        lambda_delta=0.001,
    )
)

Epoch 01 | loss=0.9148 | gate=0.032 | val_macro_f1=0.4853
Epoch 02 | loss=0.8786 | gate=0.017 | val_macro_f1=0.4853
Epoch 03 | loss=0.9490 | gate=0.026 | val_macro_f1=0.4851
Epoch 04 | loss=0.9236 | gate=0.052 | val_macro_f1=0.4851
Epoch 05 | loss=0.9008 | gate=0.091 | val_macro_f1=0.5095
Epoch 06 | loss=0.8898 | gate=0.176 | val_macro_f1=0.5211
Epoch 07 | loss=0.8556 | gate=0.239 | val_macro_f1=0.5168
Epoch 08 | loss=0.8778 | gate=0.246 | val_macro_f1=0.5291
Epoch 09 | loss=0.9007 | gate=0.254 | val_macro_f1=0.5165
Epoch 10 | loss=0.8980 | gate=0.279 | val_macro_f1=0.5291
Epoch 11 | loss=0.8896 | gate=0.292 | val_macro_f1=0.5158
Epoch 12 | loss=0.8428 | gate=0.261 | val_macro_f1=0.5221
Epoch 13 | loss=0.8734 | gate=0.293 | val_macro_f1=0.5168
Epoch 14 | loss=0.8475 | gate=0.328 | val_macro_f1=0.5270
Epoch 15 | loss=0.8359 | gate=0.325 | val_macro_f1=0.5192
Epoch 16 | loss=0.8429 | gate=0.336 | val_macro_f1=0.5213
Epoch 17 | loss=0.8261 | gate=0.345 | val_macro_f1=0.4915
Early stopping

In [116]:
residual_test_result = (
    evaluate_residual_model(
        residual_model,
        residual_test_loader,
    )
)

{
    k: v
    for k, v
    in residual_test_result.items()
    if k not in {
        "y_true",
        "y_pred",
        "alpha",
        "delta_norm",
    }
}

{'accuracy': 0.49477351916376305,
 'balanced_accuracy': 0.4741034404192298,
 'macro_f1': 0.4770544103191753,
 'weighted_f1': 0.49656151846892155}

In [117]:
residual_comparison = pd.DataFrame([
    {
        "model":
            "semantic_anchor",

        **anchor_results,
    },

    {
        "model":
            "gated_trajectory_correction",

        **{
            k:
                residual_test_result[k]

            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1",
            ]
        },
    },
])

baseline = (
    residual_comparison
    .set_index("model")
    .loc["semantic_anchor"]
)

for metric_name in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:

    residual_comparison[
        f"delta_{metric_name}"
    ] = (
        residual_comparison[
            metric_name
        ]
        - baseline[
            metric_name
        ]
    )

display(
    residual_comparison.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy,delta_balanced_accuracy,delta_macro_f1,delta_weighted_f1
0,semantic_anchor,0.5296,0.4611,0.4823,0.5278,0.0000,0.000,0.0000,0.0000
1,gated_trajectory_correction,0.4948,0.4741,0.4771,0.4966,-0.0348,0.013,-0.0052,-0.0312


In [118]:
alpha_test = (
    residual_test_result[
        "alpha"
    ]
)

pd.Series(
    alpha_test
).describe(
    percentiles=[
        .1,
        .25,
        .5,
        .75,
        .9,
        .95,
    ]
)

count    287.000000
mean       0.271927
std        0.091688
min        0.043218
10%        0.160288
25%        0.221959
50%        0.273222
75%        0.330499
90%        0.401290
95%        0.412890
max        0.449455
dtype: float64

In [119]:
final_pred = (
    residual_test_result[
        "y_pred"
    ]
)

semantic_pred = (
    anchor_test_pred
)

true_labels = (
    y_test_final
)

semantic_correct = (
    semantic_pred
    == true_labels
)

residual_correct = (
    final_pred
    == true_labels
)

rescues = (
    (~semantic_correct)
    & residual_correct
)

breaks = (
    semantic_correct
    & (~residual_correct)
)

print(
    "Semantic correct:",
    semantic_correct.sum(),
)

print(
    "Residual correct:",
    residual_correct.sum(),
)

print(
    "Rescues:",
    rescues.sum(),
)

print(
    "Breaks:",
    breaks.sum(),
)

print(
    "Net:",
    rescues.sum()
    - breaks.sum(),
)

Semantic correct: 152
Residual correct: 142
Rescues: 9
Breaks: 19
Net: -10


In [120]:
gate_analysis = pd.DataFrame({
    "alpha":
        alpha_test,

    "semantic_correct":
        semantic_correct,

    "residual_correct":
        residual_correct,

    "rescue":
        rescues,

    "break":
        breaks,

    "failure_family":
        [
            family_names[y]
            for y
            in true_labels
        ],
})

In [121]:
display(
    gate_analysis
    .groupby(
        "failure_family"
    )["alpha"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
    ])
)

,count,mean,median,std
failure_family,,,,
constraint_error,70,0.269682,0.272071,0.070200
grounding_state_error,30,0.231716,0.254799,0.086273
reasoning_value_error,11,0.247246,0.224513,0.062814
tool_use_error,38,0.221677,0.232030,0.112928
workflow_error,138,0.297611,0.306647,0.089384


In [122]:
print(
    classification_report(
        true_labels,
        final_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

                       precision    recall  f1-score   support

       workflow_error     0.5565    0.5000    0.5267       138
     constraint_error     0.5641    0.6286    0.5946        70
       tool_use_error     0.3125    0.2632    0.2857        38
grounding_state_error     0.2889    0.4333    0.3467        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.4948       287
            macro avg     0.4944    0.4741    0.4771       287
         weighted avg     0.5055    0.4948    0.4966       287



I’d name the notebook:

# `12_trajectory_representation_and_gated_correction.ipynb`

**Research question:** *Can learned agent-trajectory representations add predictive information beyond the current-message semantic representation, and can gating use that information safely?*

## Notebook result summary

This notebook investigated whether **ordered agent history** provides useful signal for failure-family classification beyond the semantic representation of the current message.

The trajectory dataset contained **4,591 historical events across 419 trajectories**, with **1,776 labeled target messages**. The canonical group-safe split was preserved: **1,489 train / 287 test targets**, with **335 train groups / 84 test groups and zero group overlap**. Historical context was substantial: targets had a median of **9 previous events**, mean **14.7**, and maximum **93**.

### 1. Learned trajectory representations contain real predictive signal

A GRU was trained over ordered trajectory events using role/tool/structural history. The trajectory representation alone was able to predict failure families, demonstrating that **agent behavior before the current message contains information about the eventual failure type**.

Using frozen embeddings with the same downstream logistic-regression evaluation:

| Representation        |   Accuracy | Balanced Acc. |   Macro F1 | Weighted F1 |
| --------------------- | ---------: | ------------: | ---------: | ----------: |
| **Semantic**          | **0.5157** |        0.4814 | **0.4903** |  **0.5215** |
| Trajectory            |     0.4774 |    **0.4851** |     0.4700 |      0.4856 |
| Semantic + trajectory |     0.4425 |        0.4431 |     0.4430 |      0.4556 |

The trajectory representation therefore came surprisingly close to semantic classification by itself and slightly exceeded semantic balanced accuracy, but remained weaker overall.

Importantly, trajectory information was not uniformly useful across classes. For example, trajectory embeddings substantially improved `tool_use_error` recall (**0.5263 vs. 0.2632 semantic**), indicating that some failure families are particularly dependent on behavioral history.

### 2. Naive fusion does not exploit the complementary signal

Simply concatenating semantic and trajectory representations made performance substantially worse:

**Macro F1: 0.4903 → 0.4430.**

Residualizing trajectory embeddings against semantic embeddings did not solve the problem either:

| Model                          |   Macro F1 |
| ------------------------------ | ---------: |
| Semantic                       | **0.4903** |
| Trajectory residual            |     0.3318 |
| Semantic + residual trajectory |     0.4394 |

Thus, the trajectory representation contains information, but much of it is correlated, noisy, or insufficiently stable relative to the semantic representation.

### 3. Learned gated correction also failed to generalize

A conservative architecture was then tested in which semantic predictions remained the anchor and trajectory information learned only a **gated residual correction**.

The gate behaved sensibly during training, initially assigning almost no trajectory influence and gradually increasing its average activation from approximately **0.03 to 0.3**.

However, on the held-out test trajectories:

| Model                       |   Accuracy | Balanced Acc. |   Macro F1 | Weighted F1 |
| --------------------------- | ---------: | ------------: | ---------: | ----------: |
| Semantic anchor             | **0.5296** |        0.4611 | **0.4823** |  **0.5278** |
| Gated trajectory correction |     0.4948 |    **0.4741** |     0.4771 |      0.4966 |
| Δ                           |    −0.0348 |   **+0.0130** |    −0.0052 |     −0.0312 |

The correction produced:

```text
Semantic correct: 152
Trajectory-corrected correct: 142

Rescues:  9
Breaks:  19
Net:    -10
```

The model therefore learned corrections that appeared useful on validation trajectories but did **not generalize to unseen trajectory groups**.

### 4. Main finding

The experiments reject the simple hypothesis that adding more sophisticated fusion or gating to the current trajectory representation will improve classification.

Instead, they support a more specific conclusion:

> **Trajectory history contains genuine complementary predictive information, but the current trajectory representation is not rich enough or stable enough for reliable fusion with semantic predictions.**

The sequence encoder currently represents history primarily through structural information such as role, tool identity, message length, error indicators, and ordering. It does not represent the **semantic content of historical messages and tool interactions**.

Consequently, it can learn patterns such as repeated tool use, action sequences, trajectory position, and interaction structure, but cannot adequately distinguish *why* two superficially similar trajectories represent different agent states.

## MoE/gating conclusion

Together with the earlier routing experiments, the notebook gives an important result about MoE.

The failure is **not evidence that MoE is useless for this task**. Rather:

> An MoE router cannot recover complementary expertise that is not sufficiently represented in its experts' features.

Across probability routing, context-aware routing, role-aware routing, trajectory-state routing, selective rescue, class-conditioned routing, and now learned gated trajectory correction, routing improvements have been small or unstable.

The oracle experiments showed that semantic and structured/trajectory experts make different mistakes, so **theoretical complementarity exists**. But learned routers have not been able to identify those cases reliably out-of-group.

This shifts the research bottleneck from:

**“How do we build a better gate?”**

to:

**“How do we learn a trajectory representation that captures the semantic state of the agent?”**

## Final notebook conclusion

I would finish the notebook with this:

> **Conclusion.** Ordered agent history is predictive of failure type: a learned trajectory encoder approaches the semantic classifier despite operating primarily on behavioral and structural sequence information, and it exhibits different class-specific strengths, particularly for tool-use failures. However, concatenation, residualization, and gated residual correction fail to convert this complementary signal into improved held-out performance. The gated correction produces more harmful overrides than successful rescues (19 vs. 9), indicating that trajectory-derived corrections do not generalize reliably across unseen trajectory groups. Combined with previous MoE routing experiments, these results suggest that routing architecture is no longer the primary bottleneck. The next research direction is therefore to enrich each historical trajectory event with semantic message/tool-result representations and learn a **text-aware trajectory state encoder** before revisiting mixture-of-experts routing.

### Next notebook

I would follow this immediately with:

**`13_text_aware_trajectory_representation.ipynb`**

Research question:

> **Does encoding the semantic content of historical agent events produce a trajectory representation that contributes information beyond the current-message semantic embedding?**

That gives you a clean progression: **routing → trajectory structure → learned trajectory representation → gated correction → semantic trajectory representation**, rather than continuing to tune gates against an expert representation that the current experiments show is the bottleneck.
